In [ ]:
import torch.nn as nn
import torch.optim as optim
import math
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import random
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Seeded runs

In [ ]:
seed = 512

# Fix seeds everywhere
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Deterministic CUDA
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Force deterministic PyTorch ops
torch.use_deterministic_algorithms(True)

### Classes

In [ ]:
class ClientDataset(Dataset):
    def __init__(self, df):
        # Ensure all features are numeric
        for col in df.columns:
            if col != "label_encoded":
                df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.fillna(0)

        self.X = torch.tensor(df.drop(["label", "label_encoded"], axis=1).values, dtype=torch.float32)
        self.y = torch.tensor(df["label_encoded"].values, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)  # output logits

    def forward(self, x):
        x = torch.relu(self.fc1(x))   # hidden layer activation
        x = torch.relu(self.fc2(x))   # hidden layer activation
        x = self.fc3(x)                  # output logits (no sigmoid here)
        return x

### Data Distribution

#### Run this for IID

In [ ]:
num_clients = 10
client_dfs = []

for i in range(1, num_clients + 1):
    df_client = pd.read_csv(f"/content/drive/MyDrive/Thesis Dataset/client{i}-IID.csv")
    df_client = df_client.sample(frac=1, random_state=42).reset_index(drop=True)  # optional shuffle
    client_dfs.append(df_client)

print(f"Loaded {num_clients} client datasets successfully.")

Loaded 10 client datasets successfully.


#### Run this for Non IID

In [ ]:
num_clients = 10
client_dfs = []

for i in range(1, num_clients + 1):
    df_client = pd.read_csv(f"/content/drive/MyDrive/Thesis Dataset/client{i}-nonIID.csv")
    df_client = df_client.sample(frac=1, random_state=42).reset_index(drop=True)  # optional shuffle
    client_dfs.append(df_client)

print(f"Loaded {num_clients} client datasets successfully.")

Loaded 10 client datasets successfully.


### Training Rounds and other variables

In [ ]:
num_rounds = 20
batch_size = 320
lr = 0.0001
local_epochs = 1

clip_norm = 0.5        # C
noise_multiplier = 1.6 # σ
max_epsilon = 32.0 # privacy budget

# APEM exclusive variables
aal = 0.05
na = 0.005
nd = 0.00375
pai = 50000
U = 2
S = 2

# NEW — maximum allowed noise
MAX_NOISE = 2.0

# Get input dimension from the first client
input_dim = client_dfs[0].drop(["label", "label_encoded"], axis=1).shape[1]

# --- Weighted BCE for imbalance ---
all_labels = pd.concat(client_dfs)["label_encoded"]
pos = (all_labels == 1).sum()
neg = (all_labels == 0).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32)

# Initialize global model
global_model = MLP(input_dim)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # changed from BCELoss to BCEWithLogitsLoss

print(f"Global model initialized with pos_weight={pos_weight.item():.4f}")

Global model initialized with pos_weight=0.0300


### DP and Accounting

In [ ]:
def clip_update(update_state_dict, C):
    total_norm = 0.0
    for key in update_state_dict:
        total_norm += torch.norm(update_state_dict[key])**2
    total_norm = torch.sqrt(total_norm)

    clip_factor = min(1.0, C / (total_norm + 1e-6))

    for key in update_state_dict:
        update_state_dict[key] *= clip_factor

    return update_state_dict

def add_gaussian_noise(update_state_dict, C, sigma):
    for key in update_state_dict:
        noise = torch.normal(
            mean=0.0,
            std=sigma * C,
            size=update_state_dict[key].shape
        )
        update_state_dict[key] += noise
    return update_state_dict

def compute_rdp(noise_multiplier, sample_rate, steps, orders):
    """Compute Renyi DP (RDP) for the Gaussian mechanism."""
    if noise_multiplier == 0:
        return float("inf")

    rdp_list = []
    for alpha in orders:
        rdp_alpha = alpha / (2 * (noise_multiplier ** 2))
        rdp_list.append(rdp_alpha)
    rdp_list = np.array(rdp_list) * steps * sample_rate
    return rdp_list

def get_epsilon(orders, rdp, delta):
    """Convert RDP to (epsilon, delta)-DP."""
    epsilons = rdp - (np.log(delta) / (orders - 1))
    idx = np.argmin(epsilons)
    return epsilons[idx], orders[idx]

### Training

In [ ]:
# --- DP & FL parameters ---
delta = 1e-5
orders = np.arange(2, 64)

# --- TRACKERS (per client) ---
client_rdp_total = [np.zeros_like(orders, dtype=float) for _ in range(num_clients)]
client_epsilons = [0.0 for _ in range(num_clients)]
client_noise = [noise_multiplier for _ in range(num_clients)]
client_rows_cumulative = [0 for _ in range(num_clients)]

client_stable_rounds = [0 for _ in range(num_clients)]
client_unstable_rounds = [0 for _ in range(num_clients)]
client_prev_eps = [0.0 for _ in range(num_clients)]

prev_global_acc = None  # for AAL accuracy-drop rule

# --- FEDERATED LEARNING LOOP ---
for r in range(num_rounds):
    print(f"\n=== FL Round {r+1} ===")
    client_updates = []

    # --- EACH CLIENT PARTICIPATES ---
    for i, client_df in enumerate(client_dfs):

        # (1) PRIVACY BUDGET CHECK — if ε > max allowed, skip client
        if client_epsilons[i] > max_epsilon:
            print(f"⚠️ Client {i+1} skipped (ε={client_epsilons[i]:.4f} > {max_epsilon})")
            continue

        # Create dataset + dataloader
        client_dataset = ClientDataset(client_df)
        loader = DataLoader(
            client_dataset,
            batch_size=batch_size,
            shuffle=True,
            worker_init_fn=lambda worker_id: np.random.seed(42 + worker_id),
            generator=torch.Generator().manual_seed(42)
        )

        # client model from global model
        client_model = MLP(input_dim)
        client_model.load_state_dict(global_model.state_dict())
        optimizer = optim.Adam(client_model.parameters(), lr=lr)

        steps = len(loader) * local_epochs
        rows_processed = 0

        # client training
        for X, y in loader:
            rows_processed += len(X)

            optimizer.zero_grad()
            preds = client_model(X)

            loss = loss_fn(preds, y)           # forward pass + loss
            loss.backward()                    # bckward pass
            torch.nn.utils.clip_grad_norm_(client_model.parameters(), clip_norm)  # Non-DP gradient clipping
            optimizer.step()                    # update client weights

        print(f"Client {i+1} processed {rows_processed} rows this round.")
        client_rows_cumulative[i] += rows_processed

        # PAI
        if client_rows_cumulative[i] >= pai:
            increments = client_rows_cumulative[i] // pai

            # if client unstable: halve NA
            effective_na = na / 2 if client_unstable_rounds[i] >= U else na

            # add noise
            client_noise[i] += increments * effective_na

            # Reset cumulative rows after each increment
            client_rows_cumulative[i] -= increments * pai

            # max noise
            client_noise[i] = min(client_noise[i], MAX_NOISE)

            print(f"📈 Client {i+1} noise increased. New noise={client_noise[i]:.4f}")

        # compute update
        update = {}
        for key in global_model.state_dict().keys():
            update[key] = client_model.state_dict()[key] - global_model.state_dict()[key]

        update = clip_update(update, clip_norm) # clipping model update
        update = add_gaussian_noise(update, clip_norm, client_noise[i]) # gaussian

        # privacy accounting rdp
        sample_rate = batch_size / len(client_dataset)
        rdp_round = compute_rdp(client_noise[i], sample_rate, steps, orders)
        client_rdp_total[i] += rdp_round

        eps, best_alpha = get_epsilon(orders, client_rdp_total[i], delta)
        client_epsilons[i] = eps

        # stability
        if abs(eps - client_prev_eps[i]) < 1e-6:
            client_stable_rounds[i] += 1
            client_unstable_rounds[i] = 0
        else:
            client_unstable_rounds[i] += 1
            client_stable_rounds[i] = 0

        client_prev_eps[i] = eps  # store ε for next round

        # decrease noise if stable
        if client_stable_rounds[i] >= S:
            client_noise[i] = max(0.01, client_noise[i] - nd)
            client_noise[i] = min(client_noise[i], MAX_NOISE)

            print(f"🔽 Client {i+1}: noise decreased. New noise={client_noise[i]:.4f}")

            client_stable_rounds[i] = 0
            client_unstable_rounds[i] = 0

        print(f"Client {i+1} trained, loss={loss.item():.4f}, ε={eps:.4f}, noise={client_noise[i]:.4f}")

        # skip clients that now exceed the privacy budget
        if eps > max_epsilon:
            print(f"⚠️ Client {i+1} exceeded ε limit.")
            continue

        # send client update
        client_updates.append(update)

    # fed avg
    if len(client_updates) == 0:
        print("No clients left. Stopping.")
        break

    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        stacked = torch.stack([upd[key] for upd in client_updates], dim=0)
        global_dict[key] += stacked.mean(0)

    global_model.load_state_dict(global_dict)

    #global evaluation
    global_model.eval()
    df_full = pd.concat(client_dfs).reset_index(drop=True)

    for col in df_full.columns:
        if col != "label_encoded":
            df_full[col] = pd.to_numeric(df_full[col], errors="coerce")
    df_full = df_full.fillna(0)

    X_global = torch.tensor(df_full.drop(["label", "label_encoded"], axis=1).values, dtype=torch.float32)
    y_global = torch.tensor(df_full["label_encoded"].values, dtype=torch.float32).unsqueeze(1)

    with torch.no_grad():
        logits_global = global_model(X_global)
        probs = torch.sigmoid(logits_global)
        preds = (probs > 0.5).float()

    accuracy = accuracy_score(y_global.numpy(), preds.numpy())
    print(f"Global accuracy after round {r+1}: {accuracy:.4f}")

    # if global accuracy loss is greater than aal, decrease all client noise by ND
    if prev_global_acc is not None:
        if (prev_global_acc - accuracy) >= aal:
            print(f"Accuracy dropped ≥ {aal}. Decreasing all noise.")

            for ci in range(num_clients):
                client_noise[ci] = max(0.01, client_noise[ci] - nd)

                client_noise[ci] = min(client_noise[ci], MAX_NOISE)
                client_stable_rounds[ci] = 0
                client_unstable_rounds[ci] = 0

                print(f"  • Client {ci+1} noise now {client_noise[ci]:.4f}")

    prev_global_acc = accuracy



=== FL Round 1 ===
Client 1 processed 37970 rows this round.
Client 1 trained, loss=8.7454, ε=3.2020, noise=1.6000
Client 2 processed 49528 rows this round.
Client 2 trained, loss=16.3636, ε=3.1995, noise=1.6000
Client 3 processed 24333 rows this round.
Client 3 trained, loss=7.6532, ε=3.2191, noise=1.6000
Client 4 processed 982353 rows this round.
📈 Client 4 noise increased. New noise=1.6950
Client 4 trained, loss=0.1570, ε=3.0055, noise=1.6950
Client 5 processed 11196 rows this round.
Client 5 trained, loss=68.8083, ε=3.1976, noise=1.6000
Client 6 processed 189732 rows this round.
📈 Client 6 noise increased. New noise=1.6150
Client 6 trained, loss=25.5977, ε=3.1647, noise=1.6150
Client 7 processed 541880 rows this round.
📈 Client 7 noise increased. New noise=1.6500
Client 7 trained, loss=0.0000, ε=3.0926, noise=1.6500
Client 8 processed 242803 rows this round.
📈 Client 8 noise increased. New noise=1.6200
Client 8 trained, loss=0.0089, ε=3.1543, noise=1.6200
Client 9 processed 33675 

### Evaluation

#### Load test dataset


In [ ]:
test_path = "/content/drive/MyDrive/Thesis Dataset/scen-testing.csv"
print(f"\nLoading external test file: {test_path}")

df_test = pd.read_csv(test_path)

for col in df_test.columns:
    if col != "label_encoded":
        df_test[col] = pd.to_numeric(df_test[col], errors="coerce")

df_test = df_test.fillna(0)
df_test = df_test.dropna(subset=["label_encoded"])  # ensure valid labels


Loading external test file: /content/drive/MyDrive/Thesis Dataset/scen-testing.csv


In [ ]:
print(df_test.shape)

(586109, 41)


#### Evaluate


In [ ]:
global_model.eval()

# --- Prepare test tensors ---
X_test = torch.tensor(
    df_test.drop(["label", "label_encoded"], axis=1).values,
    dtype=torch.float32
)

y_test = torch.tensor(
    df_test["label_encoded"].values,
    dtype=torch.float32
).unsqueeze(1)

# --- Inference ---
with torch.no_grad():
    logits = global_model(X_test)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

# --- Move to CPU for sklearn ---
y_true = y_test.cpu().numpy()
y_pred = preds.cpu().numpy()
probs_np = probs.cpu().numpy()

# --- Metrics ---
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, probs_np)
cm = confusion_matrix(y_true, y_pred)

print("\n=== Evaluation on scen-testing.csv ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("Confusion Matrix:")
print(cm)

# --- Final Privacy Accounting ---
orders = np.arange(2, 64)
sample_rate = 1.0
delta = 1e-5
steps = num_rounds

rdp = compute_rdp(noise_multiplier, sample_rate, steps, orders)
eps, best_alpha = get_epsilon(orders, rdp, delta)

print(f"\n[FINAL PRIVACY] Total ε after all rounds: {eps:.4f}, best α = {best_alpha}")


=== Evaluation on scen-testing.csv ===
Accuracy:  0.9378
Precision: 0.9959
Recall:    0.9398
F1-score:  0.9670
ROC-AUC:   0.9075
Confusion Matrix:
[[ 14848   2202]
 [ 34278 534781]]

[FINAL PRIVACY] Total ε after all rounds: 17.4752, best α = 3


In [ ]:
# --- Privacy-Utility Analysis (Accuracy-based) ---

# 1. Average privacy spent per client
active_epsilons = [eps for eps in client_epsilons if eps > 0]

if len(active_epsilons) == 0:
    avg_privacy = 0
else:
    avg_privacy = sum(active_epsilons) / len(active_epsilons)

# 2. Utility = Accuracy (as per your study)
utility = accuracy  # from test evaluation

# 3. Privacy-Utility Score
if avg_privacy == 0:
    pu_score = 0
else:
    pu_score = utility / avg_privacy

print("\n=== Privacy-Utility Analysis ===")
print(f"Average Privacy Spent (ε): {avg_privacy:.4f}")
print(f"Model Accuracy:           {utility:.4f}")
print(f"Privacy-Utility Score:    {pu_score:.6f}")


=== Privacy-Utility Analysis ===
Average Privacy Spent (ε): 16.0334
Model Accuracy:           0.9378
Privacy-Utility Score:    0.058488


#### Evaluate model

In [ ]:
global_model.eval()

# --- Prepare test tensors ---
X_test = torch.tensor(df_test.drop(["label", "label_encoded"], axis=1).values, dtype=torch.float32)
y_test = torch.tensor(df_test["label_encoded"].values, dtype=torch.float32).unsqueeze(1)

# --- Run model inference ---
with torch.no_grad():
    logits = global_model(X_test)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

y_true = y_test.numpy()
y_pred = preds.numpy()

# --- Evaluation metrics ---
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\n=== Evaluation on scen-testing.csv ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

# --- Final Privacy Accounting ---
orders = np.arange(2, 64)
sample_rate = 1.0
delta = 1e-5
steps = num_rounds

rdp = compute_rdp(noise_multiplier, sample_rate, steps, orders)
eps, best_alpha = get_epsilon(orders, rdp, delta)

print(f"\n[FINAL PRIVACY] Total ε after all rounds: {eps:.4f}, best α = {best_alpha}")